In [1]:
import json
import random
from openpyxl.reader.excel import load_workbook
from tqdm import tqdm
from openai import OpenAI
import logging

data_path = '/kaggle/input/qlu-dataset/TCM-TBOSD-train.json'
output_path = '/kaggle/working/DataNewClean2.json'
data = json.load(open(data_path, 'r', encoding='utf-8'))
print(f'总病历数：{len(data)}')

results = []
for idx, item in tqdm(enumerate(data)):
    # ['ID', '性别', '职业', '年龄', '婚姻', '病史陈述者', '发病节气', '主诉', '症状', '中医望闻切诊', '病史', '体格检查', '辅助检查', '疾病', '证型', '处方']
    #id = item['ID'] # id
    gender = item['性别'] # 性别
    #job = item['职业'] # 职业
    age = item['年龄'] # 年龄
    #marriage = item['婚姻'] # 婚姻
    #status = item['病史陈述者']
    disease_time = item['发病节气']
    chief_complaint = item['主诉']
    symptom = item['症状'] # 症状
    tcm_examination = item['中医望闻切诊'] # 中医望闻切诊
    history = item['病史'] # 病史
    physical_examination = item['体格检查'] # 体格检查
    auxiliary_examination = item['辅助检查'] # 辅助检查
    drug = item['处方'] # 处方
    writer = open(output_path, 'a', encoding='utf-8')
    

    query = f"""患者性别为{gender},年龄为{age},发病节气在{disease_time}。患者{chief_complaint}症状表现状为:{symptom}中医望闻切诊的结果为:{tcm_examination}患者病史:{history}患者体格检查结果为:{physical_examination}其他辅助检查结果:{auxiliary_examination}
    """
    prescription = item.get('处方', '')
    if isinstance(prescription, str):
        try:
            # 尝试解析为列表（如果处方是字符串形式的列表）
            prescription_list = json.loads(prescription)
        except json.JSONDecodeError:
            # 如果不是合法 JSON，假设是逗号分隔的字符串
            prescription_list = [herb.strip(" '") for herb in prescription.strip("[]").split(",") if herb.strip()]
    elif isinstance(prescription, list):
        # 如果处方已经是列表，直接使用
        prescription_list = prescription
    else:
        # 如果处方格式未知，设为空列表
        prescription_list = []
    
    # 转换为逗号分隔的字符串，去掉单引号
    cleaned_output = ",".join(herb.strip(" '") for herb in prescription_list if herb.strip())
    
    ins = """你是一名中医专家，请根据患者的信息为患者提供草药处方，要求输出中仅需要输出草药名称，不需要给出任何解释和其他信息，草药数量控制在10-15个左右。"""
    data_item = {
        'instruction': ins,
        'input': query,
        'output': cleaned_output
    }
    results.append(data_item)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)


总病历数：800


800it [00:00, 21470.58it/s]
